# 02 — Clean: Unemployment Data

Loads raw BLS data from DuckDB, standardizes dates and types, filters
annual-average placeholders, adds analysis-ready derived columns, runs
quality reports, and saves interim Parquet files to `data/interim/`.

**All cleaning runs inside DuckDB — no pandas chained operations.**

**Two tables produced:**
- `unemployment_national_clean` — national by age/sex/race + U3–U6, 1948–2026
- `unemployment_states_clean` — all states + DC, monthly, 1976–2026


In [ ]:
import sys, os
from pathlib import Path

PROJECT = Path.cwd()
while not (PROJECT / "config.yaml").exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))

from clean_quality import get_connection, quality_report, save_interim, register_source, run_sql
import pandas as pd
import duckdb

cfg = {
    "paths": {
        "data_interim": "data/interim",
        "data_processed": "data/processed",
    },
    "settings": {
        "duckdb_file": "data/project.duckdb",
        "parquet_engine": "pyarrow",
    },
}
con = get_connection(cfg)
print("Connected to DuckDB:", cfg["settings"]["duckdb_file"])
tables = [t[0] for t in con.execute("SHOW TABLES").fetchall()]
print("Tables available:", tables)


## National Unemployment Data — Cleaning

**Steps:**
1. Parse `period` ("M01"…"M12") → integer `month`; drop M13 (annual averages).
2. Derive a `date` column (first of month, as a VARCHAR ISO date for DuckDB
   compatibility across all display tools).
3. Add a `category` column grouping each series ("overall", "by_sex", "by_age",
   "by_race", "alternative_measures") for easy slicing in exploration.
4. Cast `value` to DOUBLE, keep only rows where value is not null.
5. Deduplicate on (series_id, year, month).


**New this pass:** the national pull now includes `LNS14000024` (Total, 20+ adults) — the real combined-sex adult total used by Chart 4 in `04-viz` to compare teens vs adults. It flows through this same clean pass (categorized `overall`) with no special handling.

In [ ]:
# Define category mapping for series IDs
CATEGORY_MAP = {
    "LNS14000000": "overall",
    "LNS14000024": "overall",  # Total, 20+ (adults) — real combined-sex adult total
    "LNS14000001": "by_sex",
    "LNS14000002": "by_sex",
    "LNS14000025": "by_sex",
    "LNS14000026": "by_sex",
    "LNS14000012": "by_age",
    "LNS14000036": "by_age",
    "LNS14000089": "by_age",
    "LNS14000091": "by_age",
    "LNS14000093": "by_age",
    "LNS14000095": "by_age",
    "LNS14000097": "by_age",
    "LNS14000003": "by_race",
    "LNS14000006": "by_race",
    "LNS14000009": "by_race",
    "LNS14032183": "by_race",
    "LNS14023621": "alternative",
    "LNS13327707": "alternative",
    "LNS13327708": "alternative",
    "LNS13327709": "alternative",
}

import pandas as pd

# Create the category lookup as a temp table
cat_df = pd.DataFrame([
    {"series_id": k, "category": v}
    for k, v in CATEGORY_MAP.items()
])
con.execute("DROP TABLE IF EXISTS _category_map")
con.execute("CREATE TABLE _category_map AS SELECT * FROM cat_df")

# Build the cleaned national table inside DuckDB
con.execute("DROP TABLE IF EXISTS unemployment_national_clean")
con.execute("""
CREATE TABLE unemployment_national_clean AS
SELECT
    n.series_id,
    n.label,
    COALESCE(c.category, 'other')      AS category,
    n.year,
    CAST(REPLACE(n.period, 'M', '') AS INTEGER) AS month,
    n.period_name,
    -- ISO date string for the first of the month
    printf('%04d-%02d-01', n.year, CAST(REPLACE(n.period, 'M', '') AS INTEGER)) AS date,
    TRY_CAST(n.value AS DOUBLE)        AS value
FROM unemployment_national n
LEFT JOIN _category_map c USING (series_id)
WHERE n.period != 'M13'           -- drop annual averages
  AND TRY_CAST(n.value AS DOUBLE) IS NOT NULL
ORDER BY series_id, year, month
""")

n_clean = con.execute("SELECT COUNT(*) FROM unemployment_national_clean").fetchone()[0]
print(f"unemployment_national_clean: {n_clean:,} rows")
print()
print(con.execute("SELECT * FROM unemployment_national_clean LIMIT 5").df().to_string(index=False))


In [ ]:
df_nat_clean = con.execute("SELECT * FROM unemployment_national_clean").df()
_ = quality_report(
    df_nat_clean,
    "unemployment_national_clean",
    con,
    required_columns=["series_id", "label", "category", "year", "month", "date", "value"],
)


In [ ]:
# Spot-check: well-known recession peaks should match BLS published figures
checks = con.execute("""
SELECT label, year, month, value
FROM unemployment_national_clean
WHERE (series_id = 'LNS14000000' AND year IN (1982, 2009, 2020))
   OR (series_id = 'LNS14000006' AND year = 2010)  -- Black peak ~16.0
   OR (series_id = 'LNS13327709' AND year = 2010)  -- U-6 peak ~17.2
ORDER BY series_id, year, month
""").df()

print("Recession-peak spot checks:")
print(checks.to_string(index=False))

# Distribution by category
cat_dist = con.execute("""
    SELECT category, COUNT(DISTINCT series_id) AS series, COUNT(*) AS observations,
           MIN(year) AS first_year, MAX(year) AS last_year
    FROM unemployment_national_clean
    GROUP BY category ORDER BY category
""").df()
print()
print("Coverage by category:")
print(cat_dist.to_string(index=False))


In [ ]:
from pathlib import Path
Path("data/interim").mkdir(parents=True, exist_ok=True)
df_nat_clean.to_parquet("data/interim/unemployment_national_clean.parquet", index=False)
print(f"Saved → data/interim/unemployment_national_clean.parquet  ({len(df_nat_clean):,} rows)")


## State Unemployment Data — Cleaning

**Steps:**
1. Join with `state_ref` to attach state names (already on the table as `label`).
2. Parse `period` → integer `month`; drop M13.
3. Derive `date` (ISO string, first of month).
4. Add a 2-letter `state_abbrev` column for chart labels using a FIPS lookup.
5. Cast `value` to DOUBLE, drop nulls.
6. Deduplicate on (state, year, month).


In [ ]:
# FIPS → 2-letter abbreviation lookup
FIPS_ABBREV = {
    "01": "AL", "02": "AK", "04": "AZ", "05": "AR", "06": "CA",
    "08": "CO", "09": "CT", "10": "DE", "11": "DC", "12": "FL",
    "13": "GA", "15": "HI", "16": "ID", "17": "IL", "18": "IN",
    "19": "IA", "20": "KS", "21": "KY", "22": "LA", "23": "ME",
    "24": "MD", "25": "MA", "26": "MI", "27": "MN", "28": "MS",
    "29": "MO", "30": "MT", "31": "NE", "32": "NV", "33": "NH",
    "34": "NJ", "35": "NM", "36": "NY", "37": "NC", "38": "ND",
    "39": "OH", "40": "OK", "41": "OR", "42": "PA", "44": "RI",
    "45": "SC", "46": "SD", "47": "TN", "48": "TX", "49": "UT",
    "50": "VT", "51": "VA", "53": "WA", "54": "WV", "55": "WI",
    "56": "WY",
}

abbrev_df = pd.DataFrame([
    {"series_id": f"LASST{fips}0000000000003", "state_abbrev": abbr}
    for fips, abbr in FIPS_ABBREV.items()
])
con.execute("DROP TABLE IF EXISTS _abbrev_map")
con.execute("CREATE TABLE _abbrev_map AS SELECT * FROM abbrev_df")

con.execute("DROP TABLE IF EXISTS unemployment_states_clean")
con.execute("""
CREATE TABLE unemployment_states_clean AS
SELECT
    s.series_id,
    s.label                                              AS state_name,
    COALESCE(a.state_abbrev, '?')                       AS state_abbrev,
    s.year,
    CAST(REPLACE(s.period, 'M', '') AS INTEGER)         AS month,
    s.period_name,
    printf('%04d-%02d-01', s.year,
           CAST(REPLACE(s.period, 'M', '') AS INTEGER)) AS date,
    TRY_CAST(s.value AS DOUBLE)                         AS value
FROM unemployment_states s
LEFT JOIN _abbrev_map a USING (series_id)
WHERE s.period != 'M13'
  AND TRY_CAST(s.value AS DOUBLE) IS NOT NULL
ORDER BY series_id, year, month
""")

n_states = con.execute("SELECT COUNT(*) FROM unemployment_states_clean").fetchone()[0]
print(f"unemployment_states_clean: {n_states:,} rows")
print()
print(con.execute("SELECT * FROM unemployment_states_clean LIMIT 5").df().to_string(index=False))


In [ ]:
df_states_clean = con.execute("SELECT * FROM unemployment_states_clean").df()
_ = quality_report(
    df_states_clean,
    "unemployment_states_clean",
    con,
    required_columns=["series_id", "state_name", "state_abbrev", "year", "month", "date", "value"],
)


In [ ]:
# Spot-check: current rankings + known outlier states
recent_year = con.execute("SELECT MAX(year) FROM unemployment_states_clean").fetchone()[0]
recent_month = con.execute(
    f"SELECT MAX(month) FROM unemployment_states_clean WHERE year = {recent_year}"
).fetchone()[0]

ranked = con.execute(f"""
    SELECT state_name, state_abbrev, value
    FROM unemployment_states_clean
    WHERE year = {recent_year} AND month = {recent_month}
    ORDER BY value DESC
""").df()
print(f"State rankings — {recent_year}-{recent_month:02d} (most recent):")
print(ranked.to_string(index=False))


In [ ]:
df_states_clean.to_parquet("data/interim/unemployment_states_clean.parquet", index=False)
print(f"Saved → data/interim/unemployment_states_clean.parquet  ({len(df_states_clean):,} rows)")


## Unemployment by Education × Race — Cleaning (companion table)

Builds the tidy DuckDB table **`unemployment_by_education_race`** behind the
companion race-gap chart in `04-viz` (the gap holds at every education level).

**Source:** BLS CPS annual table **cpsaat07** (25+, annual average, NSA), cached
verbatim in `data/raw/cpsaat07_2024.pdf` by `01-ingest`. We re-parse that raw PDF
here with `parse_cpsaat07()` (word-position parse) so the clean step is
reproducible from the untouched raw file.

**Standardized education tiers (4), least → most educated:**
1. Less than high school
2. High school, no college
3. Some college or associate degree
4. Bachelor's degree and higher

**Collapse decision (documented):** cpsaat07 splits tier 3 into
"Some college, no degree" + "Associate degree", and tier 4 into "Bachelor's only"
+ "Advanced degree". We take the source's own **TOTAL** column for each of the two
higher tiers rather than summing the sub-columns — BLS already publishes the
combined rate, so there is no re-derivation and no rounding drift. The sub-columns
are ignored (handled in the parser). This yields exactly the owner's four-tier
grouping.

**Grain reminder (keep separate from Charts 1–6):** annual average, **25+**,
**not seasonally adjusted** — a different series from the monthly SA 16+ `LN`
data. Never co-plot or splice.

**QC:** exactly 8 rows (2 races × 4 tiers), **no nulls** in `unemployment_rate`
for White/Black across the four tiers, and every rate a plausible 0–30 percent.
The table is registered in `_sources` with the 25+/annual/NSA methodology + the
series-break note. Interim saved to `data/interim/`.

In [ ]:
# Re-parse the cached raw cpsaat07 PDF and build the tidy companion table.
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "src"))
from ingest import parse_cpsaat07

CPSAAT07_YEAR = 2024
_pdf = Path("data/raw") / f"cpsaat07_{CPSAAT07_YEAR}.pdf"
assert _pdf.exists(), f"missing raw PDF {_pdf} — run 01-ingest first"

df_edu_race = parse_cpsaat07(_pdf, CPSAAT07_YEAR)

# Explicit tier ordering (least → most educated) carried as a helper column so
# the chart renders clusters in order.
EDU_ORDER = [
    "Less than high school",
    "High school, no college",
    "Some college or associate degree",
    "Bachelor's degree and higher",
]
order_map = {lvl: i for i, lvl in enumerate(EDU_ORDER)}
df_edu_race["education_order"] = df_edu_race["education_level"].map(order_map)

# Build the tidy DuckDB table.
con.execute("DROP TABLE IF EXISTS unemployment_by_education_race")
con.execute("""
CREATE TABLE unemployment_by_education_race AS
SELECT
    CAST(year AS INTEGER)           AS year,
    race,
    education_level,
    CAST(education_order AS INTEGER) AS education_order,
    CAST(unemployment_rate AS DOUBLE) AS unemployment_rate
FROM df_edu_race
ORDER BY year, education_order, race
""")

# ---- Quality checks ----
n_rows = con.execute("SELECT COUNT(*) FROM unemployment_by_education_race").fetchone()[0]
n_null = con.execute(
    "SELECT COUNT(*) FROM unemployment_by_education_race WHERE unemployment_rate IS NULL"
).fetchone()[0]
tiers = {r[0] for r in con.execute(
    "SELECT DISTINCT education_level FROM unemployment_by_education_race"
).fetchall()}
races = {r[0] for r in con.execute(
    "SELECT DISTINCT race FROM unemployment_by_education_race"
).fetchall()}

assert n_rows == 8, f"expected 8 rows (2 races × 4 tiers), got {n_rows}"
assert n_null == 0, f"nulls in unemployment_rate: {n_null}"
assert tiers == set(EDU_ORDER), f"unexpected education tiers: {tiers}"
assert races == {"White", "Black"}, f"unexpected races: {races}"

print(f"unemployment_by_education_race: {n_rows} rows, {n_null} nulls — QC OK")
print()
print(con.execute("""
    SELECT education_level, race, unemployment_rate
    FROM unemployment_by_education_race
    ORDER BY education_order, race
""").df().to_string(index=False))

# ---- Provenance ----
register_source(
    con,
    "unemployment_by_education_race",
    name="BLS CPS — Unemployment by Educational Attainment and Race (annual, 25+)",
    url=f"https://www.bls.gov/cps/data/aa{CPSAAT07_YEAR}/cpsaat07.pdf",
    license="Public domain (U.S. government work)",
    notes=(
        "Annual table cpsaat07, parsed from PDF with pdfplumber by word position. "
        "Race × education cross-tab of the U-3 unemployment rate; White + Black, "
        "four education tiers (two higher tiers use the published TOTAL column, "
        "sub-splits ignored). Backs the companion race-gap chart in 04-viz."
    ),
    methodology=(
        "BLS CPS household survey averaged to annual figures. U-3 unemployment "
        "rate (same definition as the project's other series) but restricted to "
        "the 25-and-over population and reported as an ANNUAL AVERAGE, NOT "
        "seasonally adjusted. Education and race are self-reported."
    ),
    series_breaks=(
        "DIFFERENT GRAIN from charts 1-6: annual / 25+ / NSA (cpsaat07) vs the "
        "monthly seasonally-adjusted 16+ LN series used elsewhere — never co-plot "
        "or splice the two. Education-attainment categories are stable across "
        "recent years; the Some-college/Associate and Bachelor's/Advanced "
        "sub-splits are published but we use the TOTAL for each higher tier."
    ),
    retrieved="2026-10-05",
)
print("\nSource registered in _sources.")

# ---- Interim parquet ----
_interim = Path("data/interim") / "unemployment_by_education_race.parquet"
_interim.parent.mkdir(parents=True, exist_ok=True)
con.execute(
    f"COPY unemployment_by_education_race TO '{_interim}' (FORMAT PARQUET)"
)
print(f"Saved interim → {_interim}")

## Summary

| Table | Rows | Notes |
|---|---|---|
| `unemployment_national_clean` | ~13.7k | Dates parsed, category added, M13 dropped |
| `unemployment_states_clean` | ~30.9k | Dates parsed, state_abbrev added, M13 dropped |

Both tables persisted in `data/project.duckdb` and saved to `data/interim/` as Parquet.

**Next:** `04-viz.ipynb` — explore headline rate trends, racial/gender gaps,
state patterns, and U3 vs U6. Decide what story to tell and which charts to build.


## Cleanup

In [ ]:
con.close()
print("Connection closed.")
